In [ ]:
# Setup: Kaggle API + dataset download
!pip install -q kaggle
import os
from google.colab import files

print('Upload your kaggle.json:')
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d xray_data
print('Dataset ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping

DATA_ROOT = '/content/xray_data/chest_xray'
TRAIN_PATH = os.path.join(DATA_ROOT, 'train')
EVAL_PATH = os.path.join(DATA_ROOT, 'test')

INPUT_SIZE = (224, 224)
BATCH = 32

# Augmentation for training only
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    horizontal_flip=True
)
eval_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_PATH, target_size=INPUT_SIZE, batch_size=BATCH, class_mode='binary'
)
eval_data = eval_gen.flow_from_directory(
    EVAL_PATH, target_size=INPUT_SIZE, batch_size=BATCH, class_mode='binary', shuffle=False
)
print(train_data.class_indices)

In [ ]:
# Model 1: Custom CNN (with batch norm)
scratch_model = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

scratch_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
                      loss='binary_crossentropy', metrics=['accuracy'])
scratch_model.summary()

stopper_cnn = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

print('\n--- Training scratch CNN ---')
hist_scratch = scratch_model.fit(
    train_data, epochs=12, validation_data=eval_data, callbacks=[stopper_cnn]
)

In [ ]:
# Model 2: Transfer learning with frozen MobileNetV2 backbone
backbone = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
backbone.trainable = False

h = GlobalAveragePooling2D()(backbone.output)
h = Dense(256, activation='relu')(h)
h = Dropout(0.4)(h)
out = Dense(1, activation='sigmoid')(h)

transfer_model = Model(inputs=backbone.input, outputs=out)
transfer_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                       loss='binary_crossentropy', metrics=['accuracy'])

stopper_tl = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

print('\n--- Training MobileNetV2 transfer model ---')
hist_transfer = transfer_model.fit(
    train_data, epochs=12, validation_data=eval_data, callbacks=[stopper_tl]
)

In [ ]:
# Compare validation curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_scratch.history['val_accuracy'], label='Scratch CNN')
axes[0].plot(hist_transfer.history['val_accuracy'], label='MobileNetV2')
axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(hist_scratch.history['val_loss'], label='Scratch CNN')
axes[1].plot(hist_transfer.history['val_loss'], label='MobileNetV2')
axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix + report for both models
from sklearn.metrics import classification_report, confusion_matrix

for name, mdl in [('Scratch CNN', scratch_model), ('MobileNetV2', transfer_model)]:
    probs = mdl.predict(eval_data, verbose=0).ravel()
    preds = (probs > 0.5).astype(int)
    print(f'=== {name} ===')
    print(confusion_matrix(eval_data.classes, preds))
    print(classification_report(eval_data.classes, preds, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
# Single-image prediction
from tensorflow.keras.preprocessing import image

def classify_xray(path, model):
    img = image.load_img(path, target_size=INPUT_SIZE)
    arr = np.expand_dims(image.img_to_array(img) / 255.0, axis=0)

    p = float(model.predict(arr, verbose=0)[0][0])
    label = 'PNEUMONIA' if p > 0.5 else 'NORMAL'
    conf = p if p > 0.5 else 1 - p

    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{label} ({conf*100:.1f}% confidence)')
    plt.show()

pneu_dir = os.path.join(EVAL_PATH, 'PNEUMONIA')
norm_dir = os.path.join(EVAL_PATH, 'NORMAL')

classify_xray(os.path.join(pneu_dir, sorted(os.listdir(pneu_dir))[0]), transfer_model)
classify_xray(os.path.join(norm_dir, sorted(os.listdir(norm_dir))[0]), transfer_model)